# Character-level GPT on `input.txt`

Trains four models on `input.txt`, one after another, and writes **`results.txt`**: a summary table, each
run's loss curve and **10 generated outputs per run**. Output *N* uses the same random seed and prompt in
every run, so the runs can be compared side by side.

| run | model |
|---|---|
| `1_bigram` | `bigram.py` as it is: the next character depends only on the current one (baseline) |
| `2_trainhere_tinytransformer` | the `TinyTransformer` from `trainhere.ipynb` with its original settings |
| `3_gpt_small` | improved GPT, about the same size and training budget as run 2 |
| `4_gpt_medium` | improved GPT, 3x bigger: the main model |

**How long is an output?** The tokenizer is character-level, so 1 token = 1 character. The length limit is
the `max_new_tokens` argument of `generate()` (500 in `gpt.py`, 100 and 50 in `trainhere.ipynb`), counted in
characters, so 500 tokens is roughly 90 words. A second limit, `block_size`, is the context window: how many
previous characters the model can look at when it picks the next one (256 in `gpt.py`, 12 in `trainhere.ipynb`).

**Changes compared to `gpt.py` and `trainhere.ipynb`**
- sampling runs in `eval()` mode under `torch.no_grad()` (`gpt.py` and `trainhere.ipynb` sampled with dropout still on)
- `temperature` and `top_k` options for sampling
- all attention heads in one fused op (`F.scaled_dot_product_attention`) instead of a Python loop over heads
- AdamW with weight decay, learning-rate warmup + cosine decay, gradient clipping, GELU, GPT-2 style init
- batches are gathered with one indexing op instead of a Python loop
- the best-validation checkpoint of every run is saved in `runs/`; the last cell reloads one without retraining
- vs `trainhere.ipynb`: context 12 -> 128, feed-forward width 2048 (the `nn.TransformerEncoderLayer` default,
  16x `n_embd`) -> 4x `n_embd`, post-norm -> pre-norm, constant lr 1e-4 -> 1e-3 decaying to 1e-4

On a 6-core laptop CPU all four runs take about 1.5-2 hours (a CUDA GPU is used automatically if present).
Progress is also appended to `runs/train_log.txt`.

In [1]:
import datetime, math, os, sys, time
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(os.cpu_count())          # all logical cores: ~20% faster than torch's default here (lower it to keep the PC responsive)
OUT_DIR = 'runs'                             # checkpoints + live training log
RESULTS_PATH = 'results.txt'                   # summary, loss curves and 10 outputs per run
LOG_PATH = os.path.join(OUT_DIR, 'train_log.txt')
os.makedirs(OUT_DIR, exist_ok=True)
open(LOG_PATH, 'w').close()

def log(msg=''):
    """print, and append to runs/train_log.txt so progress can be followed from outside the notebook"""
    print(msg, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(msg + '\n')

log(f'python {sys.version.split()[0]} | torch {torch.__version__} | {device} | {torch.get_num_threads()} threads | {os.getcwd()}')

python 3.12.3 | torch 2.4.1+cpu | cpu | 12 threads | C:\Users\Amit\Desktop\Coding\llm


In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(set(text))                      # vocabulary: every distinct character in the file
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]        # string -> list of ints
decode = lambda l: ''.join(itos[i] for i in l) # list of ints -> string

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))                       # first 90% train, last 10% validation
train_data, val_data = data[:n], data[n:]
chars_per_word = len(text) / len(text.split())
log(f'{len(text):,} characters | vocab {vocab_size} | train {len(train_data):,} | val {len(val_data):,}')

def get_batch(split, batch_size, block_size):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    offsets = torch.arange(block_size)
    x = d[ix[:, None] + offsets]               # (B, T) windows, gathered in one op
    y = d[ix[:, None] + offsets + 1]           # targets: the same windows shifted by one character
    return x.to(device), y.to(device)

1,115,394 characters | vocab 65 | train 1,003,854 | val 111,540


In [3]:
class BigramLanguageModel(nn.Module):
    """bigram.py: the logits for the next character are read straight out of a (vocab x vocab) table"""

    def __init__(self, vocab_size):
        super().__init__()
        self.block_size = 1                    # only the current character matters
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)   # (B, T, vocab)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


class TinyTransformer(nn.Module):
    """trainhere.ipynb's model, unchanged. nn.TransformerEncoderLayer defaults to post-norm,
    dropout 0.1 and dim_feedforward=2048, i.e. 16x n_embd=128."""

    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=2, block_size=64):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.pos_embedding_table = nn.Embedding(block_size, n_embd)
        encoder_layer = nn.TransformerEncoderLayer(d_model=n_embd, nhead=n_head, batch_first=True)
        self.blocks = nn.TransformerEncoder(encoder_layer, num_layers=n_layer)
        self.ln_f = nn.LayerNorm(n_embd)
        self.fc = nn.Linear(n_embd, vocab_size)
        self.block_size = block_size

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.pos_embedding_table(torch.arange(T, device=idx.device)).unsqueeze(0)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=idx.device), diagonal=1)  # causal mask
        x = self.blocks(tok_emb + pos_emb, mask=mask)
        logits = self.fc(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


class CausalSelfAttention(nn.Module):
    """All heads at once: one matmul makes q, k, v for every head, then PyTorch's fused causal attention.
    Same math as gpt.py's Head + MultiHeadAttention, without the Python loop over heads."""

    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.dropout = dropout
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_size)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)   # heads side by side again
        return self.resid_dropout(self.proj(y))


class Block(nn.Module):
    """pre-norm transformer block: attention (characters look at each other), then an MLP"""

    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    """gpt.py's GPTLanguageModel with the changes listed at the top of the notebook"""

    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)
        # GPT-2 init: shrink the layers that write into the residual stream, so the sum of the
        # 2 * n_layer residual branches starts out at a sensible scale
        for block in self.blocks:
            for w in (block.attn.proj.weight, block.mlp[2].weight):
                nn.init.normal_(w, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.token_embedding_table(idx) + self.position_embedding_table(pos))
        x = self.ln_f(self.blocks(x))
        if targets is None:
            return self.lm_head(x[:, [-1], :]), None   # sampling only needs the last position
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


MODELS = {'BigramLanguageModel': BigramLanguageModel, 'TinyTransformer': TinyTransformer, 'GPT': GPT}

In [4]:
@torch.no_grad()
def estimate_loss(model, batch_size, block_size, eval_iters):
    """average loss over eval_iters random batches of each split, with dropout off"""
    model.eval()
    out = {}
    for split in ('train', 'val'):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size, block_size)
            losses[k] = model(X, Y)[1].item()
        out[split] = losses.mean().item()
    model.train()
    return out


def make_optimizer(model, cfg):
    if 'weight_decay' not in cfg:
        return torch.optim.AdamW(model.parameters(), lr=cfg['lr'])  # what bigram.py and trainhere.ipynb do
    # weight decay on matrices and embeddings only, not on biases and LayerNorm weights
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    groups = [{'params': decay, 'weight_decay': cfg['weight_decay']}, {'params': no_decay, 'weight_decay': 0.0}]
    return torch.optim.AdamW(groups, lr=cfg['lr'], betas=cfg['betas'])


def lr_at(it, cfg):
    """linear warmup, then cosine decay from lr down to min_lr (runs without min_lr keep a constant lr)"""
    if 'min_lr' not in cfg:
        return cfg['lr']
    if it < cfg['warmup_iters']:
        return cfg['lr'] * (it + 1) / cfg['warmup_iters']
    progress = min(1.0, (it - cfg['warmup_iters']) / max(1, cfg['max_iters'] - cfg['warmup_iters']))
    return cfg['min_lr'] + 0.5 * (1 + math.cos(math.pi * progress)) * (cfg['lr'] - cfg['min_lr'])


def describe_optimizer(cfg):
    if 'min_lr' not in cfg:
        return f"AdamW, constant lr {cfg['lr']:g} (torch defaults otherwise)"
    return (f"AdamW, lr {cfg['lr']:g} -> {cfg['min_lr']:g} (warmup {cfg['warmup_iters']}, cosine), "
            f"weight decay {cfg['weight_decay']}, betas {cfg['betas']}, grad clip {cfg['grad_clip']}")


@torch.no_grad()
def generate(model, prompt='\n', max_new_tokens=500, temperature=1.0, top_k=None, seed=None):
    """Returns prompt + max_new_tokens sampled characters.
    max_new_tokens is the output length limit, counted in characters (1 token = 1 character here)."""
    model.eval()                                          # dropout off while sampling
    g = torch.Generator(device=device)
    if seed is None:
        g.seed()
    else:
        g.manual_seed(seed)
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.block_size:])     # the model only sees the last block_size characters
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('inf')
        probs = F.softmax(logits, dim=-1)
        idx = torch.cat((idx, torch.multinomial(probs, num_samples=1, generator=g)), dim=1)
    return decode(idx[0].tolist())


def train_run(cfg):
    torch.manual_seed(1337)
    model = MODELS[cfg['model']](vocab_size, **cfg['model_args']).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    optimizer = make_optimizer(model, cfg)
    B, T, max_iters = cfg['batch_size'], cfg['block_size'], cfg['max_iters']
    ckpt_path = os.path.join(OUT_DIR, cfg['name'] + '.pt')
    log(f"\n=== {cfg['name']}: {n_params:,} parameters | {max_iters:,} iters x batch {B} x context {T} ===")

    history, best = [], {'val': float('inf'), 'iter': 0}
    t0 = time.time()
    for it in range(max_iters + 1):
        lr = lr_at(it, cfg)
        for group in optimizer.param_groups:
            group['lr'] = lr

        if it % cfg['eval_interval'] == 0 or it == max_iters:
            losses = estimate_loss(model, B, T, cfg['eval_iters'])
            history.append((it, losses['train'], losses['val']))
            if losses['val'] < best['val']:           # keep the weights with the best validation loss
                best = {'val': losses['val'], 'iter': it}
                torch.save({'model': cfg['model'], 'model_args': cfg['model_args'], 'chars': chars,
                            'state_dict': model.state_dict(), 'iter': it, 'val_loss': losses['val']}, ckpt_path)
            elapsed = time.time() - t0
            eta = f", ~{elapsed / it * (max_iters - it) / 60:.0f} min left" if 0 < it < max_iters else ''
            log(f"[{cfg['name']}] step {it:>6,}/{max_iters:,} | train {losses['train']:.4f} | val {losses['val']:.4f} "
                f"| lr {lr:.1e} | {elapsed / 60:.1f} min{eta}")
        if it == max_iters:
            break

        xb, yb = get_batch('train', B, T)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if cfg.get('grad_clip'):
            nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
        optimizer.step()
    train_time = time.time() - t0

    # measure and sample with the best-validation weights, which are not necessarily the last ones
    model.load_state_dict(torch.load(ckpt_path, weights_only=True)['state_dict'])
    torch.manual_seed(0)
    final = estimate_loss(model, B, T, FINAL_EVAL_ITERS)
    log(f"[{cfg['name']}] trained in {train_time / 60:.1f} min | best val {best['val']:.4f} at step {best['iter']:,} "
        f"| best checkpoint on {FINAL_EVAL_ITERS} batches: train {final['train']:.4f}, val {final['val']:.4f}")
    return model, dict(n_params=n_params, history=history, best=best, final=final,
                       train_time=train_time, ckpt=ckpt_path)


def summary_text(results):
    lines = [f'SUMMARY (val loss of the best checkpoint over {FINAL_EVAL_ITERS} batches, lower is better)',
             f"{'run':<30}{'params':>11}{'context':>9}{'iters':>9}{'chars seen':>12}{'train time':>12}{'val loss':>10}"]
    for cfg, st, _ in results:
        seen = cfg['max_iters'] * cfg['batch_size'] * cfg['block_size']
        lines.append(f"{cfg['name']:<30}{st['n_params']:>11,}{cfg['block_size']:>9}{cfg['max_iters']:>9,}"
                     f"{seen / 1e6:>11.1f}M{st['train_time'] / 60:>10.1f} m{st['final']['val']:>10.4f}")
    for cfg in RUNS[len(results):]:
        lines.append(f"{cfg['name']:<30}  (not finished yet)")
    return '\n'.join(lines)


def write_results(results):
    L = ['Character-level language models trained on input.txt',
         f"written {datetime.datetime.now():%Y-%m-%d %H:%M} | torch {torch.__version__} on {device} ({torch.get_num_threads()} threads)",
         f"data: {len(text):,} characters, vocabulary = {vocab_size} distinct characters, 90% train / 10% validation",
         '',
         'HOW TO READ THIS',
         f"- 1 token = 1 character. Each output is {SAMPLE_CHARS} generated characters (max_new_tokens={SAMPLE_CHARS}), "
         f"about {SAMPLE_CHARS / chars_per_word:.0f} words, plus the prompt.",
         '- context (block_size) = how many previous characters the model sees when it picks the next one.',
         f"- sampling: temperature {TEMPERATURE}, top_k {TOP_K}, dropout off. Output N has the same seed and prompt in every run.",
         f"- loss = cross-entropy per character on the 10% of text the model never trained on. "
         f"Random guessing = ln({vocab_size}) = {math.log(vocab_size):.2f}.",
         '',
         summary_text(results)]
    for i, (cfg, st, samples) in enumerate(results, 1):
        seen = cfg['max_iters'] * cfg['batch_size'] * cfg['block_size']
        L += ['', '', '=' * 100,
              f"RUN {i}/{len(RUNS)}: {cfg['name']}",
              cfg['desc'],
              '=' * 100,
              f"parameters    : {st['n_params']:,}",
              f"training      : {cfg['max_iters']:,} iters x batch {cfg['batch_size']} x context {cfg['block_size']} "
              f"= {seen / 1e6:.1f}M characters (~{seen / len(train_data):.0f} passes over the training text)",
              f"optimizer     : {describe_optimizer(cfg)}",
              f"training time : {st['train_time'] / 60:.1f} min",
              f"best val loss : {st['best']['val']:.4f} at step {st['best']['iter']:,} (checkpoint {st['ckpt']})",
              f"final loss    : train {st['final']['train']:.4f} | val {st['final']['val']:.4f} "
              f"(best checkpoint, {FINAL_EVAL_ITERS} batches)",
              'loss curve    : step -> train / val']
        L += [f"  {it:>8,} -> {tr:.4f} / {va:.4f}" for it, tr, va in st['history']]
        for j, (seed, prompt, out) in enumerate(samples, 1):
            L += ['', f"----- output {j}/{len(samples)} | seed {seed} | prompt {prompt!r} -----", out.lstrip('\n')]
    with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(L) + '\n')

## Run configuration
Each run is one dict. `max_iters` = optimizer steps, `block_size` = context length in characters,
`SAMPLE_CHARS` = length of every generated output.

In [5]:
SAMPLE_CHARS = 500       # length of every output, in characters (max_new_tokens)
TEMPERATURE = 0.8        # < 1: safer, more repetitive text; 1.0: sample the model's raw distribution
TOP_K = None             # e.g. 10: only ever pick among the 10 most likely next characters
FINAL_EVAL_ITERS = 200   # batches for the final, more precise loss estimate of each run
# the same 10 (seed, prompt) pairs for every run, so the outputs can be compared across runs
SAMPLES = [(seed, '\n') for seed in range(1, 6)] + [
    (6, 'ROMEO:\n'), (7, 'JULIET:\n'), (8, 'KING RICHARD III:\n'), (9, 'First Citizen:\n'), (10, 'MENENIUS:\n')]

GPT_RECIPE = dict(lr=1e-3, min_lr=1e-4, warmup_iters=100, weight_decay=0.1, betas=(0.9, 0.99), grad_clip=1.0)
TRAINHERE_EPOCH = len(train_data) // (16 * 12)   # trainhere.ipynb's "epoch": len(train) // (batch_size * block_size)

RUNS = [
    dict(name='1_bigram',
         desc='bigram.py as it is: the next character depends only on the current character (baseline)',
         model='BigramLanguageModel', model_args={},
         batch_size=32, block_size=8, max_iters=30_000, eval_interval=3_000, eval_iters=200, lr=1e-2),
    dict(name='2_trainhere_tinytransformer',
         desc='trainhere.ipynb TinyTransformer with its original settings: 2 layers x 4 heads x 128 dim, feed-forward 2048, '
              'context 12, batch 16, constant lr 1e-4, 10 "epochs"',
         model='TinyTransformer', model_args=dict(n_embd=128, n_head=4, n_layer=2, block_size=12),
         batch_size=16, block_size=12, max_iters=10 * TRAINHERE_EPOCH, eval_interval=TRAINHERE_EPOCH,
         eval_iters=200, lr=1e-4),
    dict(name='3_gpt_small',
         desc='improved GPT, small: 4 layers x 4 heads x 128 dim, context 128 (about the size and training budget of run 2)',
         model='GPT', model_args=dict(block_size=128, n_layer=4, n_head=4, n_embd=128, dropout=0.1),
         batch_size=32, block_size=128, max_iters=3_000, eval_interval=500, eval_iters=100, **GPT_RECIPE),
    dict(name='4_gpt_medium',
         desc='improved GPT, medium: 6 layers x 6 heads x 192 dim, context 128 (the main model)',
         model='GPT', model_args=dict(block_size=128, n_layer=6, n_head=6, n_embd=192, dropout=0.1),
         batch_size=32, block_size=128, max_iters=5_000, eval_interval=500, eval_iters=50, **GPT_RECIPE),
]

## Train every run, sample 10 outputs from each, write `results.txt`
`results.txt` is rewritten after every run, so finished runs are kept even if a later one is interrupted.

In [6]:
results = []                                   # (config, stats, samples) of every finished run
t_all = time.time()
for cfg in RUNS:
    model, stats = train_run(cfg)
    t = time.time()
    samples = [(seed, prompt, generate(model, prompt, SAMPLE_CHARS, TEMPERATURE, TOP_K, seed))
               for seed, prompt in SAMPLES]
    log(f"[{cfg['name']}] sampled {len(samples)} outputs x {SAMPLE_CHARS} characters in {time.time() - t:.0f}s")
    results.append((cfg, stats, samples))
    write_results(results)
    print(f"\n--- {cfg['name']}, output 1 of {len(samples)} ---\n{samples[0][2].lstrip()}\n")
log(f'\nall {len(RUNS)} runs finished in {(time.time() - t_all) / 60:.1f} min -> {RESULTS_PATH}')
print(summary_text(results))


=== 1_bigram: 4,225 parameters | 30,000 iters x batch 32 x context 8 ===


[1_bigram] step      0/30,000 | train 4.7305 | val 4.7241 | lr 1.0e-02 | 0.0 min


[1_bigram] step  3,000/30,000 | train 2.4608 | val 2.4861 | lr 1.0e-02 | 0.1 min, ~1 min left


[1_bigram] step  6,000/30,000 | train 2.4610 | val 2.4898 | lr 1.0e-02 | 0.1 min, ~0 min left


[1_bigram] step  9,000/30,000 | train 2.4564 | val 2.4943 | lr 1.0e-02 | 0.2 min, ~0 min left


...

[4_gpt_medium] step  5,000/5,000 | train 1.1668 | val 1.4681 | lr 1.0e-04 | 57.8 min


[4_gpt_medium] trained in 57.8 min | best val 1.4665 at step 4,000 | best checkpoint on 200 batches: train 1.1986, val 1.4663


[4_gpt_medium] sampled 10 outputs x 500 characters in 59s



--- 4_gpt_medium, output 1 of 10 ---
DUKE OF YORK:
Let me above the mourn be disposed with him,
That circumstance at thee are in the news.

BUCKINGHAM:
The confess is no more more than thou shalt straight.

DUKE OF AUMERLE:
Not off I have serviced him that offend it.

ROMEO:
No, inducing thy case that is forfeit as lived,
The soldier discourse of the truth
Of the prince and bones, the boy of the poor sea-son.

HENRY BOLINGBROKE:
As not this manage to bed, the bad of Oxford.

BUCKINGHAM:
Stay, he is not so, yet a common of all.

WA


all 4 runs finished in 452.6 min -> results.txt


SUMMARY (val loss of the best checkpoint over 200 batches, lower is better)
run                                params  context    iters  chars seen  train time  val loss
1_bigram                            4,225        8   30,000        7.7M       0.6 m    2.4897
2_trainhere_tinytransformer     1,204,545       12   52,280       10.0M      25.6 m    1.7847
3_gpt_small                       824,897      128    3,000       12.3M     364.4 m    1.6098
4_gpt_medium                    2,715,713      128    5,000       20.5M      57.8 m    1.4663


## Reuse a trained model without retraining
Every run saved its best checkpoint in `runs/`. This loads one and samples from it; change the path,
`prompt`, `max_new_tokens` (output length in characters) or `temperature` as you like.

In [7]:
def load_model(path):
    ckpt = torch.load(path, map_location=device, weights_only=True)
    assert ckpt['chars'] == chars, 'checkpoint was trained on a different vocabulary'
    model = MODELS[ckpt['model']](vocab_size, **ckpt['model_args']).to(device)
    model.load_state_dict(ckpt['state_dict'])
    return model

model = load_model(os.path.join(OUT_DIR, '4_gpt_medium.pt'))
print(generate(model, prompt='ROMEO:\n', max_new_tokens=300, temperature=0.8, seed=42))

ROMEO:
Marcius shake haste; and is like of thee thy while,
And heavy we have banish'd wounds me to this world.

Second Murderer:
I may then I have too based at our sin feasts,
That should speak by my holy wife.

First Senator:
Come, be endured, spring and a friend of tear,
That had ever the mowest for the 
